# `LLMToolSelectorMiddleware`

Middleware that uses a language model to select the most relevant tools before calling the agent's main model.

When an agent has many tools, the middleware first asks a selection model which tools are relevant to the latest user message. It then replaces the request's bound `BaseTool` list with the selected subset, while preserving configured always-included tools and provider-specific tool dictionaries.

This can reduce tool-schema token usage and help the main model focus on a smaller set of tools.

- Bases: `AgentMiddleware[AgentState[ResponseT], ContextT, ResponseT]`

## Constructor

```python
LLMToolSelectorMiddleware(
    *,
    model: str | BaseChatModel | None = None,
    system_prompt: str = DEFAULT_SYSTEM_PROMPT,
    max_tools: int | None = None,
    always_include: list[str] | None = None
)
```

## Parameters

* `model` — Model used to select tools.
  * Default: `None`
  * When `None`, the middleware uses the model from the current `ModelRequest`.
  * A string model identifier is initialized using `init_chat_model`.
  * An initialized `BaseChatModel` is used directly.

* `system_prompt` — Instructions sent to the tool-selection model.
  * Default: `DEFAULT_SYSTEM_PROMPT`
  * When `max_tools` is configured, extra ordering and truncation instructions are appended internally.

* `max_tools` — Maximum number of model-selected tools to retain.
  * Default: `None`
  * `None` means there is no selection limit.
  * When the model returns more names, only its first `max_tools` valid, unique selections are accepted.
  * Tools listed in `always_include` do not count against this limit.
  * Provider-specific tool dictionaries do not count against this limit.

* `always_include` — Names of `BaseTool` objects that must always remain available.
  * Default: `None`, stored internally as an empty list.
  * Every configured name must exist among the request's bound `BaseTool` instances.
  * These tools are excluded from the selector model's candidate list.
  * They are appended after the selected `BaseTool` objects.

## Attributes

* `system_prompt` — Base prompt used by the selection model.
* `max_tools` — Maximum number of model-selected tools, or `None`.
* `always_include` — List of tool names that bypass model selection.
* `model` — Dedicated selection model, or `None` when the request's main model should be used.

## Default System Prompt

```python
DEFAULT_SYSTEM_PROMPT = (
    "Your goal is to select the most relevant tools "
    "for answering the user's query."
)
```

When `max_tools` is set, the middleware appends an instruction equivalent to:

```text
IMPORTANT: List the tool names in order of relevance,
with the most relevant first. If you exceed the maximum
number of tools, only the first <max_tools> will be used.
```

# Tool Categories

The request may contain two kinds of tool definitions.

## `BaseTool` Instances

These are eligible for:

* LLM-based selection.
* `always_include` validation.
* Removal from the request when not selected.

## Provider-Specific Tool Dictionaries

Dictionary-based tools are:

* Excluded from the selection candidates.
* Excluded from `always_include` validation.
* Never removed by this middleware.
* Appended to the final request after all selected and always-included `BaseTool` objects.

Example:

```python
request.tools = [
    search_tool,
    calculator_tool,
    {"type": "web_search_preview"},
]
```

The model may select between `search_tool` and `calculator_tool`, but:

```python
{"type": "web_search_preview"}
```

is preserved automatically.

# Internal `_SelectionRequest`

Dataclass containing the prepared inputs for one selection-model call.

```python
@dataclass
class _SelectionRequest:
    available_tools: list[BaseTool]
    system_message: str
    last_user_message: HumanMessage
    model: BaseChatModel
    valid_tool_names: list[str]
```

## Fields

* `available_tools` — Candidate tools after excluding `always_include`.
* `system_message` — Selection prompt, including any `max_tools` instruction.
* `last_user_message` — Most recent `HumanMessage` from the request.
* `model` — Dedicated selector model or the request's main model.
* `valid_tool_names` — Names accepted in the structured selection response.

# Methods

## 1. `_prepare_selection_request`

Prepares all inputs needed for the selection-model call.

```python
_prepare_selection_request(
    self,
    request: ModelRequest[ContextT]
) -> _SelectionRequest | None
```

## Behaviour

The method:

1. Returns `None` when the request has no tools.
2. Removes provider-specific dictionaries from the candidate list.
3. Validates every `always_include` name against the bound `BaseTool` instances.
4. Excludes always-included tools from the selection candidates.
5. Returns `None` when no selectable `BaseTool` remains.
6. Appends `max_tools` instructions to the system prompt when configured.
7. Searches backward for the latest `HumanMessage`.
8. Chooses the dedicated selector model or the request's main model.
9. Returns an internal `_SelectionRequest`.

### No Selection Needed

The method returns `None` when:

```text
request.tools is empty
```

or when:

```text
all bound BaseTool objects are in always_include
```

or when the request contains only provider-specific tool dictionaries and no always-included name requires validation.

The main model is then called with the original request unchanged.

### Latest User Message

Only the most recent `HumanMessage` is sent to the selector model.

The selector does not receive:

* The full conversation history.
* The request's existing system message.
* Previous AI messages.
* Previous tool results.

It receives:

```python
[
    {
        "role": "system",
        "content": selection_request.system_message,
    },
    selection_request.last_user_message,
]
```

### Missing User Message

If selectable tools exist but no `HumanMessage` is found, the method raises:

```python
AssertionError(
    "No user message found in request messages"
)
```

### Missing Always-Included Tool

If a configured name is not present among the bound `BaseTool` objects, it raises `ValueError`.

Example message:

```text
Tools in always_include not found in request: ['audit'].
Available tools: ['calculator', 'search']
```

## 2. `_process_selection_response`

Validates the selection result and creates a filtered `ModelRequest`.

```python
_process_selection_response(
    self,
    response: dict[str, Any],
    available_tools: list[BaseTool],
    valid_tool_names: list[str],
    request: ModelRequest[ContextT]
) -> ModelRequest[ContextT]
```

## Behaviour

The method:

1. Reads `response["tools"]`.
2. Rejects names outside `valid_tool_names`.
3. Removes duplicate selections.
4. Applies `max_tools` using the model's returned order.
5. Retrieves the selected tools from `available_tools`.
6. Appends the always-included tools.
7. Appends provider-specific tool dictionaries.
8. Returns `request.override(tools=...)`.

### Duplicate Selections

Repeated names are added only once.

For example:

```python
{
    "tools": [
        "search",
        "search",
        "calculator",
    ]
}
```

becomes:

```python
[
    "search",
    "calculator",
]
```

before the request is filtered.

### Invalid Selection

Any invalid name causes the entire processing step to fail.

```python
ValueError(
    "Model selected invalid tools: ['unknown_tool']"
)
```

Valid selections returned alongside an invalid name are not used because the exception is raised.

### Final Tool Ordering

`max_tools` is applied according to the order of names returned by the selection model.

However, the selected `BaseTool` objects are collected using:

```python
[
    tool
    for tool in available_tools
    if tool.name in selected_tool_names
]
```

Therefore, their final order follows the original `available_tools` order, not necessarily the ranking order returned by the selector model.

The complete final order is:

```text
1. Selected BaseTool objects in original request order
2. always_include BaseTool objects in original request order
3. Provider-specific tool dictionaries in original request order
```

## 3. `wrap_model_call`

Runs synchronous tool selection before invoking the main model.

```python
wrap_model_call(
    self,
    request: ModelRequest[ContextT],
    handler: Callable[
        [ModelRequest[ContextT]],
        ModelResponse[ResponseT]
    ]
) -> ModelResponse[ResponseT] | AIMessage
```

## Behaviour

1. Calls `_prepare_selection_request`.
2. Passes the original request directly to `handler` when selection is unnecessary.
3. Builds a dynamic structured-output schema from the candidate tools.
4. Calls the selection model synchronously.
5. Verifies that the returned value is a dictionary.
6. Filters the request through `_process_selection_response`.
7. Calls the main model handler using the modified request.

The structured selector is created using:

```python
structured_model = (
    selection_request.model
    .with_structured_output(schema)
)
```

The selection call uses:

```python
response = structured_model.invoke(
    [
        {
            "role": "system",
            "content": selection_request.system_message,
        },
        selection_request.last_user_message,
    ]
)
```

## 4. `awrap_model_call`

Asynchronous version of `wrap_model_call`.

```python
async def awrap_model_call(
    self,
    request: ModelRequest[ContextT],
    handler: Callable[
        [ModelRequest[ContextT]],
        Awaitable[
            ModelResponse[ResponseT]
        ]
    ]
) -> ModelResponse[ResponseT] | AIMessage
```

It uses:

```python
response = await structured_model.ainvoke(...)
```

and then:

```python
return await handler(modified_request)
```

The preparation, validation, schema construction, filtering, and pass-through behaviour are otherwise the same as the synchronous method.

# Structured Selection Schema

The middleware dynamically constructs a schema whose valid values are the current candidate tool names.

## `_create_tool_selection_response`

Creates the `TypeAdapter` used for structured output.

```python
_create_tool_selection_response(
    tools: list[BaseTool]
) -> TypeAdapter[Any]
```

For each tool, it creates an annotated literal equivalent to:

```python
Annotated[
    Literal["search"],
    Field(
        description=search_tool.description
    )
]
```

The generated response shape is equivalent to:

```python
class ToolSelectionResponse(TypedDict):
    tools: list[
        Literal[
            "search",
            "calculator",
            "send_email",
        ]
    ]
```

The `tools` field is described as:

```text
Tools to use. Place the most relevant tools first.
```

### Empty Candidate List

The helper raises:

```python
AssertionError(
    "Invalid usage: tools must be non-empty"
)
```

when called with an empty list.

In normal middleware flow, `_prepare_selection_request` returns `None` before this helper is called with no candidates.

### Response Type Validation

Because a JSON schema is passed to `with_structured_output`, the middleware expects a dictionary.

A non-dictionary response raises:

```python
AssertionError(
    f"Expected dict response, got {type(response)}"
)
```

# `_render_tool_list`

Formats tools as a Markdown list.

```python
_render_tool_list(
    tools: list[BaseTool]
) -> str
```

Example output:

```text
- search: Search the web
- calculator: Perform arithmetic
```

At this pinned revision, the helper is defined in the module but is not called by `LLMToolSelectorMiddleware`.

# Selection Flow

```text
Receive ModelRequest
        |
        v
Are any tools present?
        |
   No --+--> Call main model unchanged
        |
       Yes
        |
        v
Separate BaseTool objects from provider dictionaries
        |
        v
Validate always_include names
        |
        v
Remove always-included tools from candidate list
        |
        v
Are selectable BaseTool objects left?
        |
   No --+--> Call main model unchanged
        |
       Yes
        |
        v
Find latest HumanMessage
        |
        v
Create dynamic structured-output schema
        |
        v
Ask selector model for relevant tool names
        |
        v
Validate, deduplicate, and limit selected names
        |
        v
Build final tools list:
selected + always included + provider dictionaries
        |
        v
Call main model with filtered request
```

# `max_tools` Behaviour

Suppose the candidate tools are:

```text
search
calculator
send_email
lookup_order
```

and the selector returns:

```python
{
    "tools": [
        "lookup_order",
        "search",
        "calculator",
    ]
}
```

With:

```python
max_tools=2
```

the accepted names are:

```text
lookup_order
search
```

If `search` is also in `always_include`, it is excluded from the candidate list and does not consume one of the two selectable slots.

The constructor does not explicitly validate `max_tools`. For example, `max_tools=0` causes all model-selected tools to be discarded while still preserving always-included and provider-specific tools.

# `always_include` Behaviour

Example:

```python
LLMToolSelectorMiddleware(
    max_tools=2,
    always_include=[
        "safety_check",
        "audit_log",
    ],
)
```

If the selector chooses two candidates, the main model may receive:

```text
2 model-selected tools
+ safety_check
+ audit_log
+ any provider-specific dictionaries
```

Therefore, the final request may contain more than `max_tools` total tools.

The selection model does not see always-included tools as candidate options because they are already guaranteed to remain available.

# Dedicated Model vs Main Model

## Use the Agent's Main Model

```python
middleware = LLMToolSelectorMiddleware()
```

For each request:

```python
selection_model = request.model
```

## Use a Separate Model

```python
middleware = LLMToolSelectorMiddleware(
    model="openai:gpt-5.4-mini"
)
```

The string is initialized once during construction.

A smaller or cheaper model may be useful because selection is an additional model call before each main-model call that contains selectable tools.

## Use an Initialized Model

```python
from langchain_openai import ChatOpenAI

selector_model = ChatOpenAI(
    model="gpt-5.4-mini"
)

middleware = LLMToolSelectorMiddleware(
    model=selector_model
)
```

The supplied model instance is used directly.

# Examples

## Limit Selection to Three Tools

```python
from langchain.agents import create_agent
from langchain.agents.middleware import (
    LLMToolSelectorMiddleware,
)

middleware = LLMToolSelectorMiddleware(
    max_tools=3
)

agent = create_agent(
    model="openai:gpt-5.5",
    tools=[
        search,
        calculator,
        send_email,
        lookup_order,
        get_weather,
    ],
    middleware=[middleware],
)
```

Before the main model runs, the selector chooses at most three candidate tools.

## Use a Smaller Selection Model

```python
middleware = LLMToolSelectorMiddleware(
    model="openai:gpt-5.4-mini",
    max_tools=2,
)
```

## Always Include Critical Tools

```python
middleware = LLMToolSelectorMiddleware(
    model="openai:gpt-5.4-mini",
    max_tools=2,
    always_include=[
        "safety_check",
        "audit_log",
    ],
)
```

The two required tools remain available even when the selector chooses two additional tools.

## Custom Selection Instructions

```python
middleware = LLMToolSelectorMiddleware(
    system_prompt=(
        "Select only tools directly required to answer "
        "the latest user request. Avoid speculative choices."
    ),
    max_tools=2,
)
```

## Preserve Provider Tools

```python
request_tools = [
    local_search_tool,
    calculator_tool,
    {
        "type": "web_search_preview",
    },
]
```

Even when neither local `BaseTool` is selected, the provider-specific dictionary remains in the request.

# Exceptions

The module may raise:

* `ValueError`
  * A name in `always_include` is not bound as a `BaseTool`.
  * The selector model returns a tool name outside the allowed candidate names.

* `AssertionError`
  * Selectable tools exist but no `HumanMessage` is present.
  * `_create_tool_selection_response` receives no tools.
  * The structured selection model returns a non-dictionary value.

Other exceptions from `init_chat_model`, `with_structured_output`, the selection-model call, or the main-model handler propagate normally.

# Source

This reference follows the pinned LangChain source:

```text
libs/langchain_v1/langchain/agents/middleware/tool_selection.py
Commit: 42f8f79293cfb7589e5bc1d74a8ae4dfd0bf15e3
```